# pipeline_bronze

Parametrized Bronze ingestion notebook — Kafka → Delta Lake.

One notebook, 20 domains. DABs passes domain-specific widgets per task.
See ADR-002 (SMT/Bronze) and ADR-003 (parametrized notebooks).

In [ ]:
# Widget defaults match payment_events; DABs overrides per domain task.
dbutils.widgets.removeAll()

dbutils.widgets.text("table_name",          "payment_events")
dbutils.widgets.text("kafka_topic",         "pg.public.payment_events")
dbutils.widgets.text("kafka_bootstrap",     "localhost:9092")
dbutils.widgets.text("schema_registry_url", "http://localhost:8081")
dbutils.widgets.text("bronze_table",        "ubereats_dev.bronze.payment_events")
dbutils.widgets.text("checkpoint_path",     "/Volumes/ubereats_dev/checkpoints/bronze/payment_events")
dbutils.widgets.text("max_offsets",         "1000")
dbutils.widgets.text("starting_offsets",    "earliest")
dbutils.widgets.text("contract_path",       "contracts/payment_events.yml")
dbutils.widgets.text("source_mode",         "kafka")
dbutils.widgets.text("volume_path",         "/Volumes/ubereats_dev/landing/kafka_export/payment_events")

In [ ]:
table_name           = dbutils.widgets.get("table_name")
kafka_topic          = dbutils.widgets.get("kafka_topic")
kafka_bootstrap      = dbutils.widgets.get("kafka_bootstrap")
schema_registry_url  = dbutils.widgets.get("schema_registry_url")
bronze_table         = dbutils.widgets.get("bronze_table")
checkpoint_path      = dbutils.widgets.get("checkpoint_path")
max_offsets          = int(dbutils.widgets.get("max_offsets"))
starting_offsets     = dbutils.widgets.get("starting_offsets")
contract_path        = dbutils.widgets.get("contract_path")
source_mode          = dbutils.widgets.get("source_mode")
volume_path          = dbutils.widgets.get("volume_path")

print(f"[bronze] table={table_name}  topic={kafka_topic}  target={bronze_table}  source_mode={source_mode}")

In [ ]:
import sys

# contracts/ package lives at project root, one level above notebooks/
sys.path.insert(0, "..")

from contracts.loader import load_contract
from contracts.spark_schema import to_create_table_ddl

contract  = load_contract(contract_path)
merge_key = contract["table"]["merge_key"]

ddl = to_create_table_ddl(contract, bronze_table)
spark.sql(ddl)

print(f"[bronze] table {bronze_table} ready  merge_key={merge_key}")
print(ddl)

In [ ]:
from pyspark.sql.functions import col

if source_mode == "kafka":
    import requests
    from pyspark.sql.avro.functions import from_avro
    from pyspark.sql.functions import expr

    # Debezium registers subjects as {topic}-value in Schema Registry
    subject  = f"{kafka_topic}-value"
    resp     = requests.get(
        f"{schema_registry_url}/subjects/{subject}/versions/latest",
        timeout=30,
    )
    resp.raise_for_status()
    avro_schema_str = resp.json()["schema"]

    print(f"[bronze] Avro schema fetched  subject={subject}")

    # Confluent wire format: 1 magic byte + 4-byte schema ID + Avro payload
    # substring(value, 6) strips the first 5 bytes (1-indexed) to get raw Avro bytes
    raw_stream = (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", kafka_bootstrap)
        .option("subscribe", kafka_topic)
        .option("startingOffsets", starting_offsets)
        .option("maxOffsetsPerTrigger", max_offsets)
        .load()
        .select(expr("substring(value, 6)").alias("avro_bytes"))
    )

    # Post-SMT Avro schema: business fields + __op + __source_ts_ms (ADR-002)
    parsed_stream = raw_stream.select(
        from_avro(col("avro_bytes"), avro_schema_str).alias("d")
    ).select("d.*")

In [ ]:
from pyspark.sql.functions import current_timestamp

def merge_to_bronze(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    # Bronze quality: reject records with null PK (on_failure: reject, scope: [bronze])
    clean_df = batch_df.filter(col(merge_key).isNotNull())
    if clean_df.isEmpty():
        return

    # _ingested_at is defined in the contract schema and added here
    final_df = clean_df.withColumn("_ingested_at", current_timestamp())

    view = f"bronze_batch_{table_name}"
    final_df.createOrReplaceTempView(view)

    # Bronze is immutable — WHEN NOT MATCHED only; existing records are never updated
    spark.sql(f"""
        MERGE INTO {bronze_table} AS t
        USING {view} AS s
        ON t.`{merge_key}` = s.`{merge_key}`
        WHEN NOT MATCHED THEN INSERT *
    """)

In [ ]:
if source_mode == "kafka":
    (
        parsed_stream
        .writeStream
        .foreachBatch(merge_to_bronze)
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .start()
        .awaitTermination()
    )
elif source_mode == "volume":
    # Free Edition: serverless compute can't reach a self-hosted Kafka broker.
    # batch_df comes from export_kafka_to_volume.py, uploaded to a Volume —
    # no checkpoint needed, MERGE INTO ... WHEN NOT MATCHED is what makes
    # re-running this idempotent either way (the checkpoint only ever
    # avoided re-reading already-consumed Kafka offsets).
    batch_df = spark.read.format("parquet").load(volume_path)
    merge_to_bronze(batch_df, 0)
    print(f"[bronze] volume mode — {batch_df.count()} rows from {volume_path}")
else:
    raise ValueError(f"Unknown source_mode: {source_mode!r}")